## Make your first Image-to-text with Gradio and Qwen2-VL model


- Image to text models output a text from a given image.

In this notebook, we will use the Qwen2-VL model which is a multimodal model that can generate text from images.


### Step 1: Install Transformers
Install the latest Transformers plus qwen-vl-utils to use Qwen2-VL.


In [ ]:
!pip install -U "transformers>=4.42.0" accelerate bitsandbytes qwen-vl-utils


### Step 2: Import dependencies
Load the processor/model classes plus PIL, Torch, and helpers.


In [ ]:
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
from PIL import Image
import torch
import requests


### Step 3: Load the Qwen2-VL model
Initialize the processor and model, then place the model on GPU for faster inference.


In [ ]:
# Follow the documentation at https://qwen2.org/vl/

model_name = "Qwen/Qwen2-VL-2B-Instruct"
processor = AutoProcessor.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"

def load_qwen2_vl(model_name: str):
    """Load Qwen2-VL with 4-bit quantization when a GPU is available."""
    if device == "cuda":
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
        model = Qwen2VLForConditionalGeneration.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.float16,
        )
    else:
        model = Qwen2VLForConditionalGeneration.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=torch.float32,
        )
    model.eval()
    return model

model = load_qwen2_vl(model_name)

# This code would take a while to run


While running this code, you can learn about Qwen2-VL from here
[Qwen2-VL](https://qwen2.org/vl/)


### Step 4: Run image-to-text on a sample
Fetch an image, build the chat prompt, preprocess inputs, generate text, and decode the output.


In [ ]:
url = "https://www.ilankelman.org/stopsigns/australia.jpg"  # click the link to see the image
# or this image
# url = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"

image_stop = Image.open(requests.get(url, stream=True).raw).convert("RGB")

# Display the image
image_stop.show()

def resize_image(image: Image.Image, max_side: int = 768) -> Image.Image:
    """Resize large images to reduce GPU memory usage."""
    width, height = image.size
    longest_side = max(width, height)
    if longest_side <= max_side:
        return image
    scale = max_side / float(longest_side)
    new_size = (int(width * scale), int(height * scale))
    return image.resize(new_size, Image.BICUBIC)

image_stop_resized = resize_image(image_stop, max_side=768)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image_stop_resized},
            {"type": "text", "text": "What is shown in this image?"},
        ],
    },
]

# Create prompt from conversation (image + text)
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Process the image and prompt
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to(device)  # send inputs to CPU/GPU

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=64,
    )

# Trim the prompt tokens from the output
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]

output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)

print(output_text)


### Step 5: Extract the assistant answer
Trim the generated tokens to keep only the assistant response.


In [ ]:
# Filter the output text to get the answer

answer = output_text[0].strip()

print(answer)


Now, let's put everything into one function and then test our function

### Step 6: Wrap inference in a function
Create a reusable function that takes an image and a prompt (TODO: finish the body).


In [ ]:
# TODO : Try to put image-2-text in gradio platform and see the output

def generate_description(
    image: Image.Image,
    prompt: str = "What is shown in this image?",
    max_new_tokens: int = 64,
    max_side: int = 768,
) -> str:
    """Generate a description of the image using Qwen2-VL."""
    if image is None:
        return "Please upload an image."

    # TODO: Wrap the script above to one function where we can input an image and get output description of a text

    return # TODO: Output text description of the image


Then serve using Gradio. `input` will be images and textbox (prompt) and output will be text (description of the text)

### Step 7: Test the function
Run a quick test with a sample image to verify the output.


In [ ]:
# Test the function that we just build
url = "https://www.ilankelman.org/stopsigns/australia.jpg" ## click on the link to see the image

image = Image.open(requests.get(url, stream=True).raw)

generate_description(
    image,
    "What is shown in this image?"
)

Note : You can use the example image from the the folder `example_images` or you can use your own image.

### Step 8: Build a Gradio demo
Create a small UI for image upload + prompt, then return the generated description using Qwen2-VL.


In [ ]:
## The output text contains the user prompt and the generated text from the model
import gradio as gr

demo = gr.Interface(
    fn=lambda img, prompt: generate_description(img, prompt, max_new_tokens=64, max_side=768),
    inputs=[
        gr.Image(type="pil"),
        gr.Textbox(label="prompt", value="What is shown in this image?", lines=3),
    ],
    outputs=[gr.Textbox(label="Description", lines=3)],
    title="Image Description using Qwen2-VL",
    description="Upload an image to get a detailed description using Qwen2-VL",
)

demo.launch()
